In [57]:
import torch
import numpy as np
import pandas as pd
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from numpy.linalg import svd
from torch.nn.utils.parametrizations import orthogonal

In [3]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [4]:
from tqdm import tqdm

In [5]:
class MLP(nn.Module):
    def __init__(self, **kwargs):
        super().__init__()
        self.layer1 = nn.Linear(kwargs["input_shape"],kwargs["hidden_shape"])
        self.layer2 = orthogonal(nn.Linear(kwargs["hidden_shape"], kwargs["hidden_shape"])) 
        self.layer3 = nn.Linear(kwargs["hidden_shape"],kwargs["input_shape"])

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

    def get_embeddings(self,x):
        x = self.layer2(self.layer1(x))
        return x

In [51]:
class Simcc(nn.Module):
    def __init__(self, inputs,adj_file,device='cpu',N=2000):
        super(Simcc, self).__init__()
        self.device = device
        self.inputs = torch.Tensor(inputs).to(self.device)
       
        adj = pd.read_csv(adj_file,index_col=0).to_numpy()
        
        pcs = PCA(64).fit_transform(inputs)
        self.pcs = torch.Tensor(pcs).to(self.device) 
        self.adj = torch.Tensor(adj).to(self.device)

        self.ind_views = [0,1]
        self.combinations = [(0, 1)]
        self.N = N
    def train(self):
        self.mlps = MLP(input_shape = self.pcs.shape[1], hidden_shape = 16).to(self.device) 
        def sc_loss(A,Y):
            return (torch.triu(torch.cdist(Y,Y))*torch.triu(A)).mean()
          
        #for i in range(self.num_views):
        self.mlps.train()
        optimizer = optim.Adam(self.mlps.parameters(), lr=1e-3)          
        for epoch in tqdm(range(self.N), desc='Training network '):
            optimizer.zero_grad()
            x_hat = self.mlps(self.pcs)
            Y1 = self.mlps.get_embeddings(self.pcs)
            loss1 = nn.MSELoss()(self.pcs,x_hat)
            loss2 = sc_loss(self.adj, Y1)
            loss=loss1+loss2
            loss.backward()
            optimizer.step()

        self.mlps.eval() 
        Y = self.mlps.get_embeddings(self.pcs)
        Y = StandardScaler().fit_transform(Y.cpu().detach().numpy())
        
        self.emb = Y

In [75]:
mat_1_df = pd.read_csv(f"/share/dichen/lungST/variables/bestk_input_matrix_0.csv",index_col=0)
adj_file = f"/share/dichen/lungST/variables/bestk_adj_matrix_0.csv"

In [76]:
row_names_1 = mat_1_df.index.tolist()

In [77]:
mat_1 = mat_1_df.to_numpy()

In [85]:
model = Simcc(mat_1,adj_file,N=1000)

In [86]:
model.train()

Training network : 100%|██████████| 1000/1000 [01:16<00:00, 13.13it/s]


In [87]:
model.emb.shape

(4330, 16)

In [88]:
emb_out = model.emb

In [89]:
emb_out

array([[-2.0926116 ,  0.02534495,  2.0687907 , ...,  0.77126706,
         0.5176279 , -1.0126987 ],
       [-2.3369339 , -1.120129  ,  3.2935853 , ...,  0.18013994,
        -0.01200384, -0.97123927],
       [ 0.12752008, -1.2272906 , -0.73966575, ...,  0.44050318,
         0.75088745, -0.30604464],
       ...,
       [ 0.1935523 , -0.4570929 ,  0.20444435, ..., -0.20210455,
         0.19893736,  0.22170165],
       [-0.0424004 , -0.6068584 , -0.03131288, ..., -0.22719446,
         0.18159862,  0.10445797],
       [ 0.29358876, -0.6307472 ,  0.05297473, ..., -0.08737049,
         0.22310422,  0.15452453]], dtype=float32)

In [90]:
df = pd.DataFrame(emb_out)

In [91]:
df.to_csv('/share/dichen/lungST/pythonOut/Simcc_output_bestK_all_emb.csv')

# validation dataset for lung tissues

In [92]:
mat_3_df = pd.read_csv(f"/share/dichen/lungST/variables/bestk_lung_10X&NL_input_matrix_0.csv",index_col=0)
adj_file3 = f"/share/dichen/lungST/variables/bestk_lung_10X&NL_adj_matrix_0.csv"

In [93]:
mat_3= mat_3_df.to_numpy()
model3 = Simcc(mat_3,adj_file3,N=1000)

In [94]:
model3.train()

Training network : 100%|██████████| 1000/1000 [00:03<00:00, 289.43it/s]


In [95]:
pd.DataFrame(model3.emb).to_csv('/share/dichen/lungST/pythonOut/Simcc_output_emb_lung_10X&NL_bestk.csv')

In [96]:
model3.emb

array([[-0.13555753, -0.00731774, -0.10440644, ...,  0.2562903 ,
         0.1452252 , -0.46422407],
       [ 0.15265675, -0.44207364, -0.2562908 , ...,  1.1894472 ,
        -0.1118654 ,  0.51859754],
       [ 1.9643422 ,  2.1505806 , -0.59856623, ..., -0.6637142 ,
         0.05040533, -0.18247111],
       ...,
       [-0.06213813,  0.08571296,  0.00279298, ...,  0.16859005,
         0.08290522, -0.26450405],
       [-0.05389486,  0.08377188,  0.00510135, ...,  0.19459789,
         0.09181133, -0.24038208],
       [-0.07071436,  0.07444248, -0.00506633, ...,  0.19105132,
         0.06987458, -0.30142018]], dtype=float32)

# Pancancer

In [97]:
mat_4_df = pd.read_csv(f"/share/dichen/lungST/variables/bestk_panCancer_input_matrix_0.csv",index_col=0)
adj_file4 = f"/share/dichen/lungST/variables/bestk_panCancer_adj_matrix_0.csv"

In [101]:
mat_4= mat_4_df.to_numpy()
model4 = Simcc(mat_4,adj_file4)

In [102]:
model4.train()

Training network : 100%|██████████| 2000/2000 [00:10<00:00, 185.26it/s]


In [103]:
pd.DataFrame(model4.emb).to_csv('/share/dichen/lungST/pythonOut/Simcc_output_emb_panCancer_bestk.csv')

In [104]:
model4.emb

array([[-0.5201245 ,  0.02667733, -0.8110449 , ...,  0.23641607,
         0.05058603,  0.28049687],
       [ 0.15489864, -0.54308015, -0.6159186 , ...,  1.3541517 ,
        -0.22562826, -0.51597697],
       [-0.29965836, -0.3527093 , -0.37436754, ...,  0.9658072 ,
         0.01122531, -0.38514984],
       ...,
       [-0.31747046,  0.17596677,  0.5537093 , ..., -0.35648367,
        -0.58584565, -1.5469903 ],
       [-0.16925353,  0.44737172, -0.18100704, ..., -0.446175  ,
        -0.23365559, -0.60701686],
       [-0.2723143 ,  0.37813497, -0.35120147, ...,  0.26882145,
        -0.32410297,  0.23212251]], dtype=float32)